In [ ]:
# 03_admin_staff_metrics.ipynb
import pandas as pd
from datetime import datetime, timezone

from utils.io_utils import should_run, save_state, mark_state_ran, write_output_json, read_csv_to_df
from utils.metrics_utils import calculate_efficiency_score_vectorized, rank_staff

DATA_DIR = "data_exports"
STATE_PATH = "state/03_state.json"
OUT_PATH = "outputs/admin_staff/staff_metrics.json"

input_files = [
    f"{DATA_DIR}/visits.csv",
    f"{DATA_DIR}/queue_events.csv",
    f"{DATA_DIR}/staff_service_log.csv",
]

run, new_state = should_run(input_files, STATE_PATH)
if not run:
    write_output_json(OUT_PATH, {"updated": False})
    raise SystemExit("No new CSV changes detected.")

# Load
visits = read_csv_to_df(f"{DATA_DIR}/visits.csv", date_cols=["timestamp"])
events = read_csv_to_df(f"{DATA_DIR}/queue_events.csv", date_cols=["event_time"])
staff_log = read_csv_to_df(f"{DATA_DIR}/staff_service_log.csv", date_cols=["start_time","end_time"])

visits["date"] = visits["timestamp"].dt.date.astype(str)
visits["week"] = visits["timestamp"].dt.strftime("%G-W%V")
visits["month"] = visits["timestamp"].dt.strftime("%Y-%m")

period = {"start": str(visits["timestamp"].dt.date.min()), "end": str(visits["timestamp"].dt.date.max())}

# Completed service sessions
completed_sessions = staff_log[staff_log["outcome"] == "completed"].copy()
completed_sessions["date"] = completed_sessions["start_time"].dt.date.astype(str)
completed_sessions["week"] = completed_sessions["start_time"].dt.strftime("%G-W%V")
completed_sessions["month"] = completed_sessions["start_time"].dt.strftime("%Y-%m")

# Customers served (counts)
served_day = completed_sessions.groupby(["staff_id","date"], as_index=False).size().rename(columns={"size":"served"})
served_week = completed_sessions.groupby(["staff_id","week"], as_index=False).size().rename(columns={"size":"served"})
served_month = completed_sessions.groupby(["staff_id","month"], as_index=False).size().rename(columns={"size":"served"})

# Avg service time
svc_time = completed_sessions.groupby("staff_id", as_index=False).agg(
    avg_service_time_minutes=("duration_minutes","mean"),
    customers_served=("session_id","count")
)

# Avg wait time attributed to staff via "called"
called = events[events["event_type"] == "called"][["visit_id","staff_id"]].dropna()
vw = visits[["visit_id","wait_time_minutes","status"]]
staff_wait = called.merge(vw, on="visit_id", how="left")

wait_stats = staff_wait.groupby("staff_id", as_index=False).agg(
    avg_wait_time_minutes=("wait_time_minutes","mean"),
    handled=("visit_id","count")
)

# Completion rate attributed to staff via "called"
outcomes = called.merge(visits[["visit_id","status"]], on="visit_id", how="left")
outcomes["is_completed"] = (outcomes["status"] == "completed").astype(int)

completion = outcomes.groupby("staff_id", as_index=False).agg(
    completion_rate=("is_completed","mean"),
    total_handled=("visit_id","count")
)

# Merge
staff_df = svc_time.merge(wait_stats, on="staff_id", how="outer").merge(completion, on="staff_id", how="outer")
staff_df = staff_df.fillna({
    "customers_served": 0,
    "avg_service_time_minutes": 0,
    "avg_wait_time_minutes": 0,
    "completion_rate": 0
})

# Efficiency score + rank
staff_df["efficiency_score"] = calculate_efficiency_score_vectorized(
    staff_df,
    served_col="customers_served",
    avg_service_time_col="avg_service_time_minutes",
    avg_wait_time_col="avg_wait_time_minutes",
    completion_rate_col="completion_rate"
)
staff_df = rank_staff(staff_df, score_col="efficiency_score")

# Weekly trend (proxy)
weekly = completed_sessions.groupby(["staff_id","week"], as_index=False).agg(
    avg_service=("duration_minutes","mean"),
    customers=("session_id","count")
)
weekly["customers_rank"] = weekly["customers"].rank(pct=True)
weekly["dur_rank"] = weekly["avg_service"].rank(pct=True)
weekly["score"] = ((weekly["customers_rank"]*0.6 + (1-weekly["dur_rank"])*0.4) * 100).round(1)

trend_map = (weekly.sort_values(["staff_id","week"])
            .groupby("staff_id")
            .apply(lambda g: g[["week","score"]].to_dict("records"))
            .to_dict())

latest_day = completed_sessions["date"].max() if len(completed_sessions) else None
latest_week = completed_sessions["week"].max() if len(completed_sessions) else None
latest_month = completed_sessions["month"].max() if len(completed_sessions) else None

def get_counts(staff_id: str):
    day = int(served_day[(served_day.staff_id==staff_id) & (served_day.date==latest_day)]["served"].sum()) if latest_day else 0
    week = int(served_week[(served_week.staff_id==staff_id) & (served_week.week==latest_week)]["served"].sum()) if latest_week else 0
    month = int(served_month[(served_month.staff_id==staff_id) & (served_month.month==latest_month)]["served"].sum()) if latest_month else 0
    return {"day": day, "week": week, "month": month}

staff_list = []
for _, r in staff_df.iterrows():
    sid = r["staff_id"]
    staff_list.append({
      "staff_user_id": sid,
      "customers_served": get_counts(sid),
      "avg_service_time_minutes": float(round(r["avg_service_time_minutes"], 2)),
      "avg_wait_time_minutes": float(round(r["avg_wait_time_minutes"], 2)),
      "completion_rate": float(round(r["completion_rate"], 3)),
      "efficiency_score": float(round(r["efficiency_score"], 1)),
      "rank": int(r["rank"]),
      "trend_weekly": trend_map.get(sid, [])[-8:]
    })

payload = {
  "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00","Z"),
  "period": period,
  "staff": staff_list,
  "rankings": {
    "top_performers": [s["staff_user_id"] for s in staff_list[:3]],
    "needs_support": [s["staff_user_id"] for s in staff_list[-3:]] if len(staff_list) >= 3 else []
  }
}

write_output_json(OUT_PATH, payload)
save_state(STATE_PATH, mark_state_ran(new_state))

print("✅ 03 complete: outputs/admin_staff/staff_metrics.json written")
